In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, time, random
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

PARENT_FOLDER     = "/content/drive/MyDrive/OCT_Project"
IMAGE_ROOT_FOLDER = os.path.join(PARENT_FOLDER, "dataset", "ROI", "enh")
LABEL_ROOT_FOLDER = os.path.join(PARENT_FOLDER, "pred (1)", "pred")

CHECKPOINT_DIR  = os.path.join(PARENT_FOLDER, "checkpoints")
OUTPUT_MASK_DIR = "/content/masks"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

BEST_WEIGHTS = os.path.join(CHECKPOINT_DIR, "best_attention_unet.pth")

IMG_SIZE   = 256
BATCH_SIZE = 4
NUM_EPOCHS = 25
LR         = 1e-4
POS_WEIGHT = 5.0
SEED       = 42

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device:        {DEVICE}")
print(f"Parent:        {PARENT_FOLDER}  (exists: {os.path.exists(PARENT_FOLDER)})")
print(f"Image root:    {IMAGE_ROOT_FOLDER}  (exists: {os.path.exists(IMAGE_ROOT_FOLDER)})")
print(f"Label root:    {LABEL_ROOT_FOLDER}  (exists: {os.path.exists(LABEL_ROOT_FOLDER)})")
print(f"Checkpoints →  {CHECKPOINT_DIR}")
print(f"Masks →        {OUTPUT_MASK_DIR}")


Device:        cuda
Parent:        /content/drive/MyDrive/OCT_Project  (exists: True)
Image root:    /content/drive/MyDrive/OCT_Project/dataset/ROI/enh  (exists: True)
Label root:    /content/drive/MyDrive/OCT_Project/pred (1)/pred  (exists: True)
Checkpoints →  /content/drive/MyDrive/OCT_Project/checkpoints
Masks →        /content/masks


In [ ]:
from pathlib import Path
import time

label_root = Path("/content/drive/MyDrive/OCT_Project/pred (1)/pred")
image_root = Path("/content/drive/MyDrive/OCT_Project/dataset/ROI/enh")

print("Listing label root (first 5 items)...")
start = time.time()
try:
    items = list(label_root.iterdir())[:5]
    print(f"✓ {time.time()-start:.1f}s")
    for item in items:
        print(f"  {item.name}")
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")

print("\nListing image root (first 5 items)...")
start = time.time()
try:
    items = list(image_root.iterdir())[:5]
    print(f"✓ {time.time()-start:.1f}s")
    for item in items:
        print(f"  {item.name}")
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")

print("\nTrying rglob on image root (first 5 PNGs)...")
start = time.time()
try:
    items = list(image_root.rglob("*.png"))[:5]
    print(f"✓ {time.time()-start:.1f}s, found {len(items)} items")
    for item in items:
        print(f"  {item.name}")
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")

Listing label root (first 5 items)...
✓ 0.0s
  DOSM
  FOFF
  GALS
  GATS
  AMOM

Listing image root (first 5 items)...
✓ 0.0s
  GATS
  GALS
  FOFF
  COEC
  DOSM

Trying rglob on image root (first 5 PNGs)...
✓ 0.3s, found 5 items
  GATS_OG_H_389.png
  GATS_OG_H_412.png
  GATS_OG_H_418.png
  GATS_OG_H_348.png
  GATS_OG_H_366.png


In [ ]:
def collect_pairs(image_root, label_root):
    """Walk image_root for *.png, expect matching *.jpg at the same relative
    path under label_root. Patient ID = first folder under image_root."""
    image_root, label_root = Path(image_root), Path(label_root)
    all_pairs = []
    for img_path in image_root.rglob("*.png"):
        rel        = img_path.relative_to(image_root)
        patient_id = rel.parts[0]
        label_path = label_root / rel.with_suffix(".jpg")
        if label_path.exists():
            all_pairs.append({"image": img_path,
                              "label": label_path,
                              "patient": patient_id})

    print(f"Found {len(all_pairs)} matched image-label pairs.")
    if not all_pairs:
        raise RuntimeError("No pairs found — check the paths in cell 2.")

    patients = sorted({p["patient"] for p in all_pairs})
    train_pats, val_pats = train_test_split(
        patients, test_size=0.2, random_state=SEED, shuffle=True)

    train_pairs = [p for p in all_pairs if p["patient"] in train_pats]
    val_pairs   = [p for p in all_pairs if p["patient"] in val_pats]
    print(f"Patients: {len(patients)} total → {len(train_pats)} train / {len(val_pats)} val")
    print(f"Images:   {len(train_pairs)} train / {len(val_pairs)} val")
    return train_pairs, val_pairs

train_pairs, val_pairs = collect_pairs(IMAGE_ROOT_FOLDER, LABEL_ROOT_FOLDER)


Found 2327 matched image-label pairs.
Patients: 10 total → 8 train / 2 val
Images:   1853 train / 474 val


In [ ]:
class OCTDataset(Dataset):
    def __init__(self, pairs, augment=False, img_size=IMG_SIZE):
        self.pairs, self.augment, self.img_size = pairs, augment, img_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p   = self.pairs[idx]
        img  = cv2.imread(str(p["image"]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(p["label"]), cv2.IMREAD_GRAYSCALE)
        img  = cv2.resize(img,  (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        if self.augment:
            if random.random() > 0.5:
                img, mask = cv2.flip(img, 1), cv2.flip(mask, 1)
            if random.random() > 0.5:
                angle = random.uniform(-5, 5)
                M = cv2.getRotationMatrix2D((self.img_size/2, self.img_size/2), angle, 1.0)
                img  = cv2.warpAffine(img,  M, (self.img_size, self.img_size), flags=cv2.INTER_LINEAR)
                mask = cv2.warpAffine(mask, M, (self.img_size, self.img_size), flags=cv2.INTER_NEAREST)
            if random.random() > 0.5:
                img = np.clip(img.astype(np.float32) * random.uniform(0.9, 1.1), 0, 255).astype(np.uint8)

        img_t  = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t

train_loader = DataLoader(OCTDataset(train_pairs, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(OCTDataset(val_pairs, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Train batches: 464 | Val batches: 119


In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1, x1 = self.W_g(g), self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode="bilinear", align_corners=False)
        return x * self.psi(self.relu(g1 + x1))


class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        def cb(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True))
        def ub(ic, oc):
            return nn.Sequential(nn.ConvTranspose2d(ic, oc, 2, stride=2), nn.ReLU(inplace=True))

        self.e1, self.e2 = cb(in_ch, 64), cb(64, 128)
        self.e3, self.e4 = cb(128, 256),  cb(256, 512)
        self.center      = cb(512, 1024)
        self.a4 = AttentionBlock(1024, 512, 256)
        self.a3 = AttentionBlock(512,  256, 128)
        self.a2 = AttentionBlock(256,  128,  64)
        self.a1 = AttentionBlock(128,   64,  32)
        self.d4 = ub(1024+512, 512)
        self.d3 = ub(512+256,  256)
        self.d2 = ub(256+128,  128)
        self.d1 = ub(128+64,    64)
        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        orig = x.shape[2:]
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        e4 = self.e4(F.max_pool2d(e3, 2))
        c  = self.center(F.max_pool2d(e4, 2))
        a4 = self.a4(c, e4)
        d4 = self.d4(torch.cat([F.interpolate(c,  e4.shape[2:], mode="bilinear", align_corners=False), a4], dim=1))
        a3 = self.a3(d4, e3)
        d3 = self.d3(torch.cat([F.interpolate(d4, e3.shape[2:], mode="bilinear", align_corners=False), a3], dim=1))
        a2 = self.a2(d3, e2)
        d2 = self.d2(torch.cat([F.interpolate(d3, e2.shape[2:], mode="bilinear", align_corners=False), a2], dim=1))
        a1 = self.a1(d2, e1)
        d1 = self.d1(torch.cat([F.interpolate(d2, e1.shape[2:], mode="bilinear", align_corners=False), a1], dim=1))
        return F.interpolate(self.final(d1), size=orig, mode="bilinear", align_corners=False)


In [ ]:
class DiceBCELoss(nn.Module):
    """BCE-with-logits (with pos_weight for class imbalance) + soft Dice."""
    def __init__(self, pos_weight=5.0, smooth=1.0):
        super().__init__()
        self.register_buffer("pw", torch.tensor(pos_weight))
        self.smooth = smooth

    def forward(self, logits, target):
        bce   = F.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pw)
        probs = torch.sigmoid(logits)
        inter = (probs * target).sum(dim=(1,2,3))
        denom = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice  = 1 - (2*inter + self.smooth) / (denom + self.smooth)
        return bce + dice.mean()


def dice_score(logits, target, thresh=0.5, smooth=1.0):
    pred  = (torch.sigmoid(logits) > thresh).float()
    inter = (pred * target).sum(dim=(1,2,3))
    denom = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2*inter + smooth) / (denom + smooth)).mean().item()


def iou_score(logits, target, thresh=0.5, smooth=1e-6):
    pred  = (torch.sigmoid(logits) > thresh).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = (pred + target - pred*target).sum(dim=(1,2,3))
    return ((inter + smooth) / (union + smooth)).mean().item()


In [ ]:
import matplotlib.pyplot as plt

model     = AttentionUNet(in_ch=1, out_ch=1).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR)
criterion = DiceBCELoss(pos_weight=POS_WEIGHT).to(DEVICE)

best_iou = -1.0
history  = {"train_loss": [], "train_dice": [], "val_dice": [], "val_iou": []}
print(f"Training on {DEVICE} | {len(train_pairs)} train / {len(val_pairs)} val")

for epoch in range(NUM_EPOCHS):
    model.train()
    t0, t_loss, t_dice = time.time(), 0.0, 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward(); optimizer.step()
        t_loss += loss.item()
        t_dice += dice_score(logits, masks)
    t_loss /= len(train_loader); t_dice /= len(train_loader)

    model.eval()
    v_dice, v_iou = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            v_dice += dice_score(logits, masks)
            v_iou  += iou_score(logits, masks)
    v_dice /= len(val_loader); v_iou /= len(val_loader)

    dt = time.time() - t0
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | loss {t_loss:.4f} | "
          f"train Dice {t_dice:.4f} | val Dice {v_dice:.4f} | "
          f"val IoU {v_iou:.4f} | {dt:.1f}s")

    history["train_loss"].append(t_loss); history["train_dice"].append(t_dice)
    history["val_dice"].append(v_dice);   history["val_iou"].append(v_iou)

    if v_iou > best_iou:
        best_iou = v_iou
        torch.save(model.state_dict(), BEST_WEIGHTS)
        print(f"   ↳ new best (val IoU {v_iou:.4f}) → {BEST_WEIGHTS}")

print(f"\nDone. Best val IoU: {best_iou:.4f}")

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], label="train loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_dice"], label="train Dice")
ax[1].plot(history["val_dice"],   label="val Dice")
ax[1].plot(history["val_iou"],    label="val IoU")
ax[1].set_title("Metrics"); ax[1].legend()
plt.show()


In [ ]:
!pip install plotly -q

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import os, re
from pathlib import Path
import numpy as np
import cv2
import torch
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.ndimage import label as cc_label, binary_closing, binary_fill_holes
from scipy.ndimage import generate_binary_structure, iterate_structure
import matplotlib.pyplot as plt

pio.renderers.default = "colab"

image_root = Path(IMAGE_ROOT_FOLDER)
label_root = Path(LABEL_ROOT_FOLDER)

_model = AttentionUNet(in_ch=1, out_ch=1).to(DEVICE)
_model.load_state_dict(torch.load(BEST_WEIGHTS, map_location=DEVICE))
_model.eval()

CACHE = {"subpath": None, "prob": None, "raw": None, "gt": None,
         "shapes": None, "paths": None}

def _slice_idx(name):
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0

def _list_subdirs(p):
    return sorted([d.name for d in Path(p).iterdir() if d.is_dir()]) if Path(p).exists() else []

def run_inference(subpath):
    """Run the model once on a patient stack; cache raw probabilities (not
    thresholded), original images, and ground-truth masks if present."""
    pdir = image_root / subpath
    paths = sorted(pdir.glob("*.png"), key=lambda p: _slice_idx(p.name))
    if not paths:
        raise RuntimeError(f"No PNGs in {pdir}")

    probs, raws, gts, shapes = [], [], [], []
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        h0, w0 = img.shape
        shapes.append((h0, w0))
        raws.append(img)

        img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        t = torch.from_numpy(img_r).float().unsqueeze(0).unsqueeze(0).to(DEVICE) / 255.0
        with torch.no_grad():
            prob = torch.sigmoid(_model(t)).squeeze().cpu().numpy()   # 0..1, IMG_SIZE²
        probs.append(prob)

        rel = p.relative_to(image_root)
        lp  = label_root / rel.with_suffix(".jpg")
        if lp.exists():
            g = cv2.imread(str(lp), cv2.IMREAD_GRAYSCALE)
            gts.append((g > 127).astype(np.uint8))
        else:
            gts.append(None)

    CACHE.update(subpath=subpath, prob=np.array(probs), raw=raws,
                 gt=gts, shapes=shapes, paths=paths)

def build_volume(threshold):
    """Threshold the cached probability volume into a binary (Z,H,W) volume.
    Cheap — no model involved."""
    return (CACHE["prob"] > threshold).astype(np.uint8)

def compute_roi(volume_3d, close_radius=15):
    struct = iterate_structure(generate_binary_structure(2, 1), close_radius)
    roi = np.zeros_like(volume_3d, dtype=bool)
    for z in range(volume_3d.shape[0]):
        sl = volume_3d[z].astype(bool)
        if sl.sum():
            roi[z] = binary_fill_holes(binary_closing(sl, structure=struct))
    return roi

patients = _list_subdirs(image_root)
patient_dd = widgets.Dropdown(options=patients, description="Patient:")
eye_dd     = widgets.Dropdown(description="Eye:")
orient_dd  = widgets.Dropdown(description="Orient:")
load_btn   = widgets.Button(description="Load patient", button_style="primary",
                            icon="download")

colorby_dd = widgets.Dropdown(options=["depth", "pore size", "pore ID"],
                              value="depth", description="Colour by:")
thresh_sl  = widgets.FloatSlider(value=0.5, min=0.3, max=0.7, step=0.05,
                                 description="Threshold:", continuous_update=False)
minpore_sl = widgets.IntSlider(value=5, min=1, max=50, step=1,
                               description="Min pore:", continuous_update=False)
slice_sl   = widgets.IntSlider(value=0, min=0, max=0, step=1,
                               description="2D slice:", continuous_update=False)

status   = widgets.HTML("<i>Pick a patient and click Load.</i>")
out_3d   = widgets.Output()
out_2d   = widgets.Output()
out_bio  = widgets.Output()

def _refresh_eyes(*_):
    eyes = _list_subdirs(image_root / patient_dd.value)
    eye_dd.options = eyes
    if eyes: eye_dd.value = eyes[0]
def _refresh_orients(*_):
    if not eye_dd.value:
        orient_dd.options = []; return
    ors = _list_subdirs(image_root / patient_dd.value / eye_dd.value)
    orient_dd.options = ors
    if ors: orient_dd.value = ors[0]
patient_dd.observe(_refresh_eyes,   names="value")
eye_dd.observe(_refresh_orients,    names="value")
_refresh_eyes(); _refresh_orients()

fig3d = go.FigureWidget(
    data=[go.Scatter3d(
        x=[], y=[], z=[], mode="markers",
        marker=dict(size=2, colorscale="Viridis",
                    colorbar=dict(title="Slice Index"))
    )]
)
fig3d.update_layout(
    width=850, height=600, margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(xaxis_title="X (width)", yaxis_title="Y (height)",
               zaxis_title="Slice Index", aspectmode="data")
)

def render_3d():
    if CACHE["prob"] is None:
        return
    vol = build_volume(thresh_sl.value)
    labeled, n_raw = cc_label(vol, structure=np.ones((3,3,3)))
    sizes_raw = np.bincount(labeled.ravel())[1:] if n_raw else np.array([])
    keep = sizes_raw >= minpore_sl.value
    kept_ids = set(np.where(keep)[0] + 1)

    zz, yy, xx = np.where(vol > 0)
    pid = labeled[zz, yy, xx]
    survive = np.isin(pid, list(kept_ids))
    xx, yy, zz, pid = xx[survive], yy[survive], zz[survive], pid[survive]

    if len(xx) > 60000:
        idx = np.random.choice(len(xx), 60000, replace=False)
        xx, yy, zz, pid = xx[idx], yy[idx], zz[idx], pid[idx]

    mode = colorby_dd.value
    if mode == "depth":
        color, title, scale = zz, "Slice Index", "Viridis"
    elif mode == "pore size":
        color, title, scale = sizes_raw[pid-1], "Pore vol (vox)", "Plasma"
    else:
        color, title, scale = pid, "Pore ID", "Turbo"

    with fig3d.batch_update():
        fig3d.data[0].x = xx
        fig3d.data[0].y = yy
        fig3d.data[0].z = zz
        fig3d.data[0].marker.color = color
        fig3d.data[0].marker.colorscale = scale
        fig3d.data[0].marker.colorbar.title = title
        fig3d.layout.title = (f"3D LC Pores — {CACHE['subpath']} "
                              f"(thr={thresh_sl.value}, min={minpore_sl.value})")
def render_biomarkers():
    if CACHE["prob"] is None: return
    with out_bio:
        out_bio.clear_output(wait=True)
        vol = build_volume(thresh_sl.value)
        labeled, n_raw = cc_label(vol, structure=np.ones((3,3,3)))
        sizes_raw = np.bincount(labeled.ravel())[1:] if n_raw else np.array([])
        keep = sizes_raw >= minpore_sl.value
        sizes = sizes_raw[keep]
        roi = compute_roi(vol)
        roi_v = int(roi.sum())
        pore_in = int((vol.astype(bool) & roi).sum())
        ctvf_roi = (roi_v - pore_in) / roi_v if roi_v else float("nan")
        print("--- BIOMARKERS ---")
        print(f"CTVF (LC-ROI):           {ctvf_roi:.4f}")
        print(f"Porosity (1-CTVF):       {1-ctvf_roi:.4f}")
        print(f"Pores (≥{minpore_sl.value} vox):    {int(keep.sum())}")
        print(f"Avg pore volume:         {sizes.mean():.2f} voxels" if sizes.size else "Avg pore volume: n/a")
        if sizes.size:
            print(f"Pore-size: min={sizes.min()} median={int(np.median(sizes))} max={sizes.max()}")

def render_2d():
    if CACHE["prob"] is None: return
    with out_2d:
        out_2d.clear_output(wait=True)
        z = slice_sl.value
        h0, w0 = CACHE["shapes"][z]
        raw  = CACHE["raw"][z]
        pred = (CACHE["prob"][z] > thresh_sl.value).astype(np.uint8)
        pred = cv2.resize(pred, (w0, h0), interpolation=cv2.INTER_NEAREST)
        gt   = CACHE["gt"][z]

        fig, ax = plt.subplots(1, 3, figsize=(12, 4))
        ax[0].imshow(raw, cmap="gray");  ax[0].set_title(f"Input OCT (slice {z})")
        if gt is not None:
            ax[1].imshow(cv2.resize(gt, (w0, h0), interpolation=cv2.INTER_NEAREST), cmap="gray")
            ax[1].set_title("Ground Truth")
        else:
            ax[1].text(0.5, 0.5, "no GT", ha="center", va="center"); ax[1].set_title("Ground Truth")
        ax[2].imshow(pred, cmap="gray"); ax[2].set_title(f"Prediction (thr={thresh_sl.value})")
        for a in ax: a.axis("off")
        plt.tight_layout(); plt.show()

def refresh_all_cheap(*_):
    render_3d(); render_biomarkers(); render_2d()

def on_load(_):
    sub = f"{patient_dd.value}/{eye_dd.value}/{orient_dd.value}"
    status.value = f"<i>Running inference on {sub} …</i>"
    try:
        run_inference(sub)
    except Exception as e:
        status.value = f"<b style='color:red'>{e}</b>"; return
    slice_sl.max = CACHE["prob"].shape[0] - 1
    slice_sl.value = CACHE["prob"].shape[0] // 2
    status.value = f"<b style='color:green'>Loaded {sub} — {CACHE['prob'].shape[0]} slices.</b>"
    refresh_all_cheap()

load_btn.on_click(on_load)
colorby_dd.observe(lambda c: render_3d(),                names="value")
thresh_sl.observe(lambda c: refresh_all_cheap(),         names="value")
minpore_sl.observe(lambda c: (render_3d(), render_biomarkers()), names="value")
slice_sl.observe(lambda c: render_2d(),                  names="value")

controls = widgets.VBox([
    widgets.HBox([patient_dd, eye_dd, orient_dd, load_btn]),
    status,
    widgets.HBox([colorby_dd, thresh_sl, minpore_sl]),
])
display(widgets.VBox([
    controls,
    widgets.HBox([fig3d, out_bio]),
    widgets.HTML("<hr><b>2D slice viewer</b>"),
    slice_sl,
    out_2d,
]))

In [ ]:
import re
from pathlib import Path
import numpy as np
import cv2
import torch
from scipy.ndimage import label as cc_label

COHORT_THRESHOLD = thresh_sl.value
COHORT_MIN_PORE  = minpore_sl.value

COHORT = {"threshold": COHORT_THRESHOLD, "min_pore": COHORT_MIN_PORE, "rows": []}


def _slice_idx(name):
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def _list_subdirs(p):
    return sorted([d.name for d in Path(p).iterdir() if d.is_dir()]) if Path(p).exists() else []


def _enumerate_patient_paths(image_root):
    """Yield (subpath, [slice png paths]) for every patient/eye/orientation
    folder that contains PNG slices."""
    image_root = Path(image_root)
    for patient in _list_subdirs(image_root):
        for eye in _list_subdirs(image_root / patient):
            for orient in _list_subdirs(image_root / patient / eye):
                d = image_root / patient / eye / orient
                pngs = sorted(d.glob("*.png"), key=lambda p: _slice_idx(p.name))
                if pngs:
                    yield f"{patient}/{eye}/{orient}", pngs


def _biomarkers_for(pngs, threshold, min_pore):
    """Run the model over a slice stack and compute the report's biomarkers.
    Mirrors the report's pipeline exactly."""
    probs = []
    for p in pngs:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        t = torch.from_numpy(img_r).float().unsqueeze(0).unsqueeze(0).to(DEVICE) / 255.0
        with torch.no_grad():
            prob = torch.sigmoid(_model(t)).squeeze().cpu().numpy()
        probs.append(prob)
    probs = np.array(probs)

    vol = (probs > threshold).astype(np.uint8)
    labeled, n_raw = cc_label(vol, structure=np.ones((3, 3, 3)))
    sizes_raw = np.bincount(labeled.ravel())[1:] if n_raw else np.array([])
    keep = sizes_raw >= min_pore
    sizes = sizes_raw[keep]

    roi = compute_roi(vol)
    roi_v = int(roi.sum())
    pore_in = int((vol.astype(bool) & roi).sum())
    ctvf = (roi_v - pore_in) / roi_v if roi_v else float("nan")

    return {
        "ctvf": ctvf,
        "porosity": 1 - ctvf,
        "n_pores": int(keep.sum()),
        "avg_pv": float(sizes.mean()) if sizes.size else 0.0,
        "med_pv": int(np.median(sizes)) if sizes.size else 0,
        "max_pv": int(sizes.max()) if sizes.size else 0,
    }


def build_cohort():
    rows = []
    paths = list(_enumerate_patient_paths(IMAGE_ROOT_FOLDER))
    print(f"Scanning {len(paths)} eye(s) at threshold {COHORT_THRESHOLD}, "
          f"min pore {COHORT_MIN_PORE} ...")
    for i, (subpath, pngs) in enumerate(paths, 1):
        try:
            b = _biomarkers_for(pngs, COHORT_THRESHOLD, COHORT_MIN_PORE)
            b["subpath"] = subpath
            rows.append(b)
            print(f"  [{i}/{len(paths)}] {subpath}: porosity {b['porosity']:.3f}, "
                  f"pores {b['n_pores']}")
        except Exception as e:
            print(f"  [{i}/{len(paths)}] {subpath}: skipped ({e})")
    COHORT["rows"] = rows
    COHORT["threshold"] = COHORT_THRESHOLD
    COHORT["min_pore"] = COHORT_MIN_PORE
    print(f"\nCohort built: {len(rows)} eyes. "
          f"The report will now phrase metrics relative to this cohort.")
    return COHORT

build_cohort()

Scanning 38 eye(s) at threshold 0.7, min pore 5 ...
  [1/38] ALLV/OD/H: porosity 0.174, pores 214
  [2/38] ALLV/OD/V: porosity 0.155, pores 194
  [3/38] ALLV/OG/H: porosity 0.172, pores 324
  [4/38] ALLV/OG/V: porosity 0.152, pores 413
  [5/38] AMOM/OD/H: porosity 0.180, pores 68
  [6/38] AMOM/OD/V: porosity 0.574, pores 90
  [7/38] AMOM/OG/H: porosity 0.177, pores 354
  [8/38] AMOM/OG/V: porosity 0.180, pores 481
  [9/38] BOUJ/OD/H: porosity 0.167, pores 329
  [10/38] BOUJ/OD/V: porosity 0.228, pores 243
  [11/38] BOUJ/OG/H: porosity 0.210, pores 236
  [12/38] BOUJ/OG/V: porosity 0.219, pores 317
  [13/38] BOUM/OD/H: porosity 0.280, pores 98
  [14/38] BOUM/OD/V: porosity 0.242, pores 113
  [15/38] BOUM/OG/H: porosity 0.190, pores 216
  [16/38] BOUM/OG/V: porosity 0.184, pores 192
  [17/38] BRUN/OD/H: porosity 0.189, pores 373
  [18/38] BRUN/OD/V: porosity 0.248, pores 287
  [19/38] BRUN/OG/H: porosity 0.496, pores 104
  [20/38] BRUN/OG/V: porosity 0.205, pores 283
  [21/38] COEC/OG/H:

{'threshold': 0.7,
 'min_pore': 5,
 'rows': [{'ctvf': 0.8255663298010174,
   'porosity': 0.1744336701989826,
   'n_pores': 214,
   'avg_pv': 59.04205607476636,
   'med_pv': 28,
   'max_pv': 535,
   'subpath': 'ALLV/OD/H'},
  {'ctvf': 0.8454865111246908,
   'porosity': 0.15451348887530925,
   'n_pores': 194,
   'avg_pv': 78.95360824742268,
   'med_pv': 34,
   'max_pv': 1403,
   'subpath': 'ALLV/OD/V'},
  {'ctvf': 0.8280526595804892,
   'porosity': 0.17194734041951076,
   'n_pores': 324,
   'avg_pv': 93.52777777777777,
   'med_pv': 30,
   'max_pv': 1936,
   'subpath': 'ALLV/OG/H'},
  {'ctvf': 0.847792211151128,
   'porosity': 0.15220778884887198,
   'n_pores': 413,
   'avg_pv': 118.06537530266344,
   'med_pv': 32,
   'max_pv': 2621,
   'subpath': 'ALLV/OG/V'},
  {'ctvf': 0.8195496246872394,
   'porosity': 0.18045037531276065,
   'n_pores': 68,
   'avg_pv': 179.35294117647058,
   'med_pv': 41,
   'max_pv': 1308,
   'subpath': 'AMOM/OD/H'},
  {'ctvf': 0.42625990224169896,
   'porosity': 0.

## Results summary



In [ ]:
!pip install "kaleido==0.2.1"

In [ ]:
import io, base64
from datetime import datetime
try:
    from zoneinfo import ZoneInfo
    _IST = ZoneInfo("Asia/Kolkata")
except Exception:
    _IST = None
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle, FancyBboxPatch
from scipy.ndimage import label as cc_label
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

BLUE = "#1a5fb4"; LIGHT = "#eaf1fb"; GREY = "#444"
LMARG, RMARG = 0.06, 0.94

LH      = 0.0132
GAP_HD  = 0.0100
GAP_PAR = 0.0105
GAP_SEC = 0.0180
BODY_WRAP = 95
BULLET_WRAP = 92


def _now_str():
    dt = datetime.now(_IST) if _IST else datetime.now()
    suffix = " IST" if _IST else ""
    return dt.strftime("%d %b %Y, %I:%M %p") + suffix


def _wrap(text, width):
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur)+len(w)+1 > width: lines.append(cur); cur = w
        else: cur = (cur+" "+w).strip()
    if cur: lines.append(cur)
    return lines


def _cohort_rank(metric_key, value, higher_is_more=True):
    """Return a short '(Nth of M eyes ...)' phrase ranking `value` within the
    cohort built by the batch cell. Returns '' if no cohort is available, so
    the report still works standalone."""
    try:
        rows = COHORT.get("rows", [])
    except NameError:
        return ""
    vals = [r[metric_key] for r in rows if r.get(metric_key) is not None
            and not (isinstance(r[metric_key], float) and np.isnan(r[metric_key]))]
    m = len(vals)
    if m < 2:
        return ""

    ordered = sorted(vals, reverse=higher_is_more)

    rank = 1 + sum(1 for v in ordered if (v > value if higher_is_more else v < value))
    pct = 100.0 * (m - rank) / (m - 1)
    return f"{rank}/{m}"


def _findings(ctvf, porosity, n_pores, avg_pv, med_pv, max_pv, n_slices):
    het = "a heterogeneous pore-size population" if max_pv > 5*max(med_pv,1) \
          else "a relatively uniform pore-size population"
    return [
        f"The lamina cribrosa was reconstructed from {n_slices} OCT slices. Automated "
        f"segmentation identified {n_pores} distinct three-dimensional pores within the "
        f"connective-tissue region of interest.",
        f"Connective tissue volume fraction (CTVF) was {ctvf:.3f}, corresponding to a "
        f"porosity of {porosity:.3f} ({porosity*100:.1f}% pore space). Lower CTVF / higher "
        f"porosity indicates a more open, less dense laminar microstructure.",
        f"Pore volumes ranged from small to large (median {med_pv}, mean {avg_pv:.1f}, "
        f"maximum {max_pv} voxels), indicating {het}.",
        "The 3D reconstructions show the spatial arrangement of pores through the laminar "
        "depth, coloured by depth, by pore volume, and by individual pore identity.",
    ]


def _has_cohort():
    try:
        return len(COHORT.get("rows", [])) >= 2
    except NameError:
        return False


def _scatter3d_png(xx, yy, zz, color, scale, cbar, title):
    fig = go.Figure(go.Scatter3d(
        x=xx, y=yy, z=zz, mode="markers",
        marker=dict(size=2.5, color=color, colorscale=scale, opacity=0.85,
                    colorbar=dict(title=dict(text=cbar, side="right"),
                                  thickness=14, len=0.7, x=1.02))))
    fig.update_layout(title=dict(text=title, x=0.5, y=0.95, font=dict(size=14)),
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Slice",
                   aspectmode="data", camera=dict(eye=dict(x=1.6, y=1.6, z=0.8))),
        width=560, height=460, margin=dict(l=8, r=8, t=42, b=8), paper_bgcolor="white")
    return fig.to_image(format="png", scale=2)


def build_report(_=None):
    out.clear_output(wait=True)
    with out:
        if CACHE["prob"] is None:
            print("No patient loaded. Pick one in the dashboard and click 'Load patient'.")
            return
        subpath   = CACHE["subpath"]; threshold = thresh_sl.value
        min_pore  = minpore_sl.value; slice_z = slice_sl.value
        safe = subpath.replace("/", "_"); parts = subpath.split("/")
        eye = {"OD":"Right eye (OD)","OG":"Right eye (OD)","OS":"Left eye (OS)"}.get(
              parts[1] if len(parts)>1 else "", parts[1] if len(parts)>1 else "-")
        orient = {"H":"Horizontal","V":"Vertical"}.get(parts[2] if len(parts)>2 else "", "-")
        print(f"Building report for {subpath} ...")

        vol = (CACHE["prob"] > threshold).astype(np.uint8)
        labeled, n_raw = cc_label(vol, structure=np.ones((3,3,3)))
        sizes_raw = np.bincount(labeled.ravel())[1:] if n_raw else np.array([])
        keep = sizes_raw >= min_pore; kept = set(np.where(keep)[0]+1); sizes = sizes_raw[keep]
        zz, yy, xx = np.where(vol > 0); pid = labeled[zz, yy, xx]
        s = np.isin(pid, list(kept)); xx, yy, zz, pid = xx[s], yy[s], zz[s], pid[s]
        if len(xx) > 40000:
            idx = np.random.choice(len(xx), 40000, replace=False)
            xx, yy, zz, pid = xx[idx], yy[idx], zz[idx], pid[idx]
        pv_per_vox = sizes_raw[pid-1]
        roi = compute_roi(vol); roi_v = int(roi.sum())
        pore_in = int((vol.astype(bool) & roi).sum())
        ctvf = (roi_v-pore_in)/roi_v if roi_v else float("nan"); porosity = 1-ctvf
        n_pores = int(keep.sum())
        avg_pv = float(sizes.mean()) if sizes.size else 0.0
        med_pv = int(np.median(sizes)) if sizes.size else 0
        max_pv = int(sizes.max()) if sizes.size else 0

        png_d = _scatter3d_png(xx, yy, zz, zz, "Viridis", "Slice", "Coloured by depth")
        png_s = _scatter3d_png(xx, yy, zz, pv_per_vox, "Plasma", "Vol", "Coloured by pore size")
        png_i = _scatter3d_png(xx, yy, zz, pid, "Turbo", "ID", "Coloured by pore ID")
        h0, w0 = CACHE["shapes"][slice_z]; raw = CACHE["raw"][slice_z]
        pred = cv2.resize((CACHE["prob"][slice_z] > threshold).astype(np.uint8),
                          (w0,h0), interpolation=cv2.INTER_NEAREST)
        gt = CACHE["gt"][slice_z]; now = _now_str()

        fig = plt.figure(figsize=(8.27, 11.69))  # A4 portrait
        fig.patch.set_facecolor("white")
        ax = fig.add_axes([0,0,1,1]); ax.axis("off"); ax.set_xlim(0,1); ax.set_ylim(0,1)

        def rule(y, c=BLUE, lw=1.5):
            ax.plot([LMARG, RMARG], [y, y], color=c, lw=lw)
        def heading(y, label):
            ax.text(LMARG, y, label, fontsize=12.5, fontweight="bold", color=BLUE, va="top")
            return y - LH - GAP_HD
        def para(y, text, width=BODY_WRAP, size=10.5, bullet=False, **kw):
            lines = _wrap(text, width)
            x_text = LMARG + 0.03 if bullet else LMARG
            if bullet:
                ax.text(LMARG + 0.008, y, "•", fontsize=size, va="top")
            for ln in lines:
                ax.text(x_text, y, ln, fontsize=size, va="top", **kw)
                y -= LH
            return y

        # ---- header band ----
        ax.add_patch(Rectangle((0, 0.945), 1, 0.055, color=BLUE))
        ax.text(0.5, 0.9725, "LAMINA CRIBROSA  —  3D OCT ANALYSIS REPORT",
                ha="center", va="center", fontsize=15, fontweight="bold", color="white")

        # ---- metadata: left block + right block share the same top baseline ----
        top = 0.928
        ax.text(LMARG, top, subpath, fontsize=15, fontweight="bold", va="top")
        ax.text(LMARG, top-0.020, f"Eye: {eye}     Scan orientation: {orient}",
                fontsize=9.5, color=GREY, va="top")
        ax.text(LMARG, top-0.035, f"Slices analysed: {vol.shape[0]}      Volume: {tuple(vol.shape)}",
                fontsize=9.5, color=GREY, va="top")
        ax.text(RMARG, top, "Reported on:", ha="right", fontsize=10.5, fontweight="bold", va="top")
        ax.text(RMARG, top-0.018, now, ha="right", fontsize=9.5, color=GREY, va="top")
        ax.text(RMARG, top-0.035, "Modality: OCT", ha="right", fontsize=9.5, color=GREY, va="top")
        rule(top-0.050)

        y = top - 0.050 - GAP_SEC
        y = heading(y, "PART")
        y = para(y, "Lamina cribrosa, optic nerve head — 3D reconstruction from OCT slice stack.")
        y -= GAP_SEC

        y = heading(y, "TECHNIQUE")
        y = para(y, f"Per-slice pore segmentation by an Attention U-Net (segmentation threshold "
                 f"{threshold}); 3D volume reconstructed by stacking masks; pores under {min_pore} "
                 f"voxels excluded as speckle. CTVF computed within an LC region of interest.")
        y -= GAP_SEC

        # ---- quantitative results: clean 3-col x 2-row grid in a box ----
        y = heading(y, "QUANTITATIVE RESULTS")
        box_h = 0.086
        pad_top = 0.005                      # space from box top to first label
        ax.add_patch(FancyBboxPatch((LMARG, y-box_h), RMARG-LMARG, box_h,
                     boxstyle="round,pad=0.004", facecolor=LIGHT, edgecolor="#bcd"))
        inner_l = LMARG + 0.035
        col_x = [inner_l, inner_l + 0.300, inner_l + 0.600]
        lab1_y = y - pad_top
        val1_y = lab1_y - 0.016
        lab2_y = val1_y - 0.022
        val2_y = lab2_y - 0.016
        pairs1 = [("CTVF", f"{ctvf:.3f}"),
                  ("Porosity", f"{porosity:.3f}  ({porosity*100:.1f}%)"),
                  ("3D pores", f"{n_pores}")]
        pairs2 = [("Mean pore", f"{avg_pv:.1f} vox"),
                  ("Median pore", f"{med_pv} vox"),
                  ("Max pore", f"{max_pv} vox")]
        for (lab,val),x in zip(pairs1, col_x):
            ax.text(x, lab1_y, lab, fontsize=8.5, color=GREY, va="top")
            ax.text(x, val1_y, val, fontsize=8.5, fontweight="bold", va="top")
        for (lab,val),x in zip(pairs2, col_x):
            ax.text(x, lab2_y, lab, fontsize=8.5, color=GREY, va="top")
            ax.text(x, val2_y, val, fontsize=8.5, fontweight="bold", va="top")
        # cohort rank tokens (only if a cohort has been built); sits inside the box
        if _has_cohort():
            rank_y = val2_y - 0.016
            ranks = [_cohort_rank("ctvf", ctvf, higher_is_more=True),
                     _cohort_rank("porosity", porosity, higher_is_more=True),
                     _cohort_rank("n_pores", n_pores, higher_is_more=True)]
            for rk, x in zip(ranks, col_x):
                if rk:
                    ax.text(x, rank_y, f"cohort rank {rk}", fontsize=7.5,
                            color=BLUE, va="top", style="italic")
        y -= box_h + GAP_SEC

        # ---- findings ----
        y = heading(y, "FINDINGS")
        for f in _findings(ctvf, porosity, n_pores, avg_pv, med_pv, max_pv, vol.shape[0]):
            y = para(y, f, width=100, size=10, bullet=True)
            y -= GAP_PAR
        y -= (GAP_SEC - GAP_PAR)

        y = para(y, "Clinical correlation is recommended. Comparison with normative reference "
"data and specialist review may be considered where available.", width=110, size=9.5, style="italic")
        y -= 0.006     # minimal lead-in to the image band

        # ---- images: FIXED reserved band above the footer ----
        FOOTER_Y    = 0.050        # footer rule
        PANEL_W     = 0.285        # width of each image panel (3 across)
        PANEL_H_3D  = 0.130        # 3D panels: wide & short
        PANEL_H_2D  = 0.120        # 2D panels: a little taller
        ROW_GAP     = 0.020        # gap between rows (room for 2D titles)
        col_step    = 0.297        # horizontal stride between the 3 columns
        row2_bottom = FOOTER_Y + 0.045
        row1_bottom = row2_bottom + PANEL_H_2D + ROW_GAP
        for i, png in enumerate([png_d, png_s, png_i]):
            iax = fig.add_axes([LMARG + i*col_step, row1_bottom, PANEL_W, PANEL_H_3D]); iax.axis("off")
            iax.imshow(mpimg.imread(io.BytesIO(png), format="png"))
        titles = [f"Input OCT (slice {slice_z})", "Ground Truth", f"Prediction (thr={threshold})"]
        imgs2 = [raw, (cv2.resize(gt,(w0,h0),interpolation=cv2.INTER_NEAREST)
                       if gt is not None else np.zeros((h0,w0))), pred]
        for i, (im, t) in enumerate(zip(imgs2, titles)):
            iax = fig.add_axes([LMARG + i*col_step, row2_bottom, PANEL_W, PANEL_H_2D])
            iax.imshow(im, cmap="gray"); iax.set_title(t, fontsize=9.5); iax.axis("off")

        # ---- footer ----
        rule(FOOTER_Y)
        fy = 0.040
        for ln in _wrap("Research output only — not a diagnostic device. Measures are uncalibrated "
                        "against normative clinical data and depend on the chosen segmentation "
                        "threshold. Requires clinical correlation and expert review.", 135):
            ax.text(LMARG, fy, ln, fontsize=7.5, style="italic", color="#a00", va="top")
            fy -= 0.011
        ax.text(LMARG, fy-0.002, "**** End of Report ****", fontsize=8, color=GREY, va="top")
        ax.text(RMARG, 0.012, f"Generated: {now}", ha="right", fontsize=8, color=GREY)

        buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                                        facecolor="white"); plt.show()
        png_bytes = buf.getvalue(); b64 = base64.b64encode(png_bytes).decode()
        display(HTML(f'<a download="{safe}_report.png" '
                f'href="data:image/png;base64,{b64}" '
                f'style="display:inline-block;padding:10px 20px;margin-top:10px;background:#1a73e8;'
                f'color:white;border-radius:6px;text-decoration:none;font-family:sans-serif;">'
                f'Download report (.png)</a>'))


out = widgets.Output()
btn = widgets.Button(description="Build report", button_style="success", icon="file-text")
btn.on_click(build_report)
display(widgets.VBox([btn, out]))